# 任务一：植物分类 —— 植物背景分割 × 双强架构 5 折集成（v5）

**方案：**
1. **植物/土壤背景分割**：ExG 超绿指数 + Otsu 阈值 + 形态学 + 连通域，去除砾石/土壤背景与测试图条形码，让模型专注植株形态
2. **Stratified 5-Fold CV**：OOF 无偏评估与集成权重搜索
3. **两种强架构**：EfficientNetV2-M + ConvNeXt-Base（共 10 个模型）
4. **384×384 高分辨率** + AMP 混合精度
5. **两阶段训练**：头部热身 2 epoch → 全网络微调 13 epoch（warmup + cosine，label smoothing，MixUp）
6. **OOF 网格搜索**集成权重
7. **TTA**：384/352 × 水平翻转 = 4 形态
8. 指标：macro-F1


In [ ]:
import os, glob, random, time, copy, cv2
from collections import Counter
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms, models

cv2.setNumThreads(1)
SEED = 42
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP_ON = DEVICE == "cuda"
print("device:", DEVICE, "| torch:", torch.__version__)
if AMP_ON:
    print("gpu:", torch.cuda.get_device_name(0))

DATA_ROOT = None
for p in glob.glob("/kaggle/input/**/train", recursive=True):
    DATA_ROOT = os.path.dirname(p); break
assert DATA_ROOT is not None, "未找到 train 目录"
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
TEST_DIR  = os.path.join(DATA_ROOT, "test")
TMPL_CSV  = glob.glob("/kaggle/input/**/*.csv", recursive=True)[0]

N_FOLDS   = 5
IMG_SIZE  = 384
BATCH     = 12
NWORK     = 2
HEAD_EP   = 2
FT_EP     = 13
WARMUP_EP = 1
ARCHS     = ["effv2m", "convnextb"]
SEEDS     = [42, 7]

probe = datasets.ImageFolder(TRAIN_DIR)
CLASSES = probe.classes
ALL_FILES = [s[0] for s in probe.samples]
Y = np.array([s[1] for s in probe.samples], dtype=np.int64)
test_files = sorted(glob.glob(TEST_DIR+"/*.png") + glob.glob(TEST_DIR+"/*.jpg"))
tmpl_df = pd.read_csv(TMPL_CSV)
print("类别:", CLASSES)
print("训练:", len(ALL_FILES), "| 测试:", len(test_files), "| 模板列:", list(tmpl_df.columns))

# ---------- 背景分割 ----------
def segment_pil(pil_img):
    img = np.array(pil_img.convert("RGB"))
    r = img[:,:,0].astype(np.int16)
    g = img[:,:,1].astype(np.int16)
    b = img[:,:,2].astype(np.int16)
    exg = 2*g - r - b
    exg_n = cv2.normalize(exg, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, mask = cv2.threshold(exg_n, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    k = np.ones((5,5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k, iterations=1)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(mask)
    out = np.zeros_like(mask)
    for i in range(1, n):
        if stats[i, cv2.CC_STAT_AREA] > 0.002 * mask.size:
            out[lab == i] = 255
    res = img.copy()
    res[out == 0] = 0
    return Image.fromarray(res)

class PlantDS(Dataset):
    def __init__(self, files, targets, tf):
        self.files = files; self.targets = targets; self.tf = tf
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        img = segment_pil(Image.open(self.files[i]))
        return self.tf(img), int(self.targets[i])


In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(np.zeros(len(Y)), Y))
for k, (tr, va) in enumerate(folds):
    print(f"fold{k}: train {len(tr)} val {len(va)} | val类别计数 {np.bincount(Y[va]).tolist()}")

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

def make_transforms():
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6,1.0), ratio=(0.85,1.15)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(25),
        transforms.ColorJitter(0.25,0.25,0.25,0.05),
        transforms.RandomAutocontrast(p=0.3),
        transforms.RandomAdjustSharpness(1.5,p=0.3),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
        transforms.RandomErasing(p=0.2, scale=(0.02,0.1)),
    ])
    val_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])
    return train_tf, val_tf


In [ ]:
from torchvision.models import (
    EfficientNet_V2_M_Weights, ConvNeXt_Base_Weights)

def build_model(arch, ncls):
    if arch == "effv2m":
        m = models.efficientnet_v2_m(weights=EfficientNet_V2_M_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, ncls)
    elif arch == "convnextb":
        m = models.convnext_base(weights=ConvNeXt_Base_Weights.IMAGENET1K_V1)
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, ncls)
    else:
        raise ValueError(arch)
    return m

def head_params(model, arch):
    if arch == "effv2m":  return list(model.classifier.parameters())
    return list(model.classifier.parameters())


In [ ]:
from sklearn.metrics import f1_score

def predict_probs(model, loader):
    model.eval()
    out = []
    with torch.no_grad():
        for x, _ in loader:
            with torch.amp.autocast("cuda", enabled=AMP_ON):
                out.append(torch.softmax(model(x.to(DEVICE)),1).cpu())
    return torch.cat(out)

def train_fold(arch, fold, tr_idx, va_idx, seed):
    seed_everything(seed)
    train_tf, val_tf = make_transforms()
    train_ds = Subset(PlantDS(ALL_FILES, Y.tolist(), train_tf), tr_idx)
    val_ds   = Subset(PlantDS(ALL_FILES, Y.tolist(), val_tf), va_idx)
    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                              num_workers=NWORK, pin_memory=True, drop_last=False)
    val_loader   = DataLoader(val_ds, batch_size=BATCH, shuffle=False,
                              num_workers=NWORK, pin_memory=True)

    model = build_model(arch, len(CLASSES)).to(DEVICE)
    crit = nn.CrossEntropyLoss(label_smoothing=0.05)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_ON)

    # stage1 头部热身
    for p in model.parameters(): p.requires_grad = False
    hp = head_params(model, arch)
    for p in hp: p.requires_grad = True
    opt = optim.AdamW(hp, lr=1e-3, weight_decay=1e-4)
    for ep in range(HEAD_EP):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            with torch.amp.autocast("cuda", enabled=AMP_ON):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()

    # stage2 全网络微调
    for p in model.parameters(): p.requires_grad = True
    hids = set(id(p) for p in head_params(model, arch))
    h_p = [p for p in model.parameters() if id(p) in hids]
    b_p = [p for p in model.parameters() if id(p) not in hids]
    opt = optim.AdamW([{"params":b_p,"lr":8e-5},{"params":h_p,"lr":2.5e-4}],
                      weight_decay=1e-4)
    def lr_lambda(ep):
        if ep < WARMUP_EP: return (ep+1)/WARMUP_EP
        prog = (ep-WARMUP_EP)/max(1,FT_EP-WARMUP_EP)
        return 0.5*(1+np.cos(np.pi*prog))
    sched = optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    best_f1, best_state = 0.0, None
    for ep in range(FT_EP):
        model.train(); t0=time.time(); rl, nb = 0.0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            do_mix = np.random.rand() < 0.4
            if do_mix:
                lam = float(np.random.beta(0.2,0.2))
                perm = torch.randperm(x.size(0), device=DEVICE)
                xin = lam*x + (1-lam)*x[perm]
            else:
                xin = x; lam = 1.0
            opt.zero_grad()
            with torch.amp.autocast("cuda", enabled=AMP_ON):
                lg = model(xin)
                loss = (lam*crit(lg,y)+(1-lam)*crit(lg,y[perm])) if do_mix else crit(lg,y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            rl += loss.item(); nb += 1
        sched.step()
        vp = predict_probs(model, val_loader)
        f1 = f1_score(Y[va_idx], vp.argmax(1).tolist(), average="macro")
        if f1 > best_f1:
            best_f1 = f1; best_state = copy.deepcopy(model.state_dict())
        print(f"  f{fold} {arch} ep{ep+1}/{FT_EP} f1={f1:.4f} {time.time()-t0:.0f}s",
              flush=True)

    ckpt = f"/kaggle/working/ckpt_{arch}_f{fold}.pth"
    torch.save({"state_dict":best_state,"arch":arch}, ckpt)
    model.load_state_dict(best_state)
    val_probs = predict_probs(model, val_loader)
    del model
    torch.cuda.empty_cache()
    return ckpt, val_probs, best_f1


In [ ]:
oof = {a: np.zeros((len(Y), len(CLASSES)), dtype=np.float32) for a in ARCHS}
t0_all = time.time()
for fold, (tr_idx, va_idx) in enumerate(folds):
    for ai, arch in enumerate(ARCHS):
        print(f"===== fold{fold} {arch} =====", flush=True)
        ckpt, vp, f1 = train_fold(arch, fold, tr_idx, va_idx, SEEDS[ai])
        oof[arch][va_idx] = vp.numpy()
        print(f">>> fold{fold} {arch} best f1={f1:.4f}", flush=True)
print(f"\n总训练耗时 {(time.time()-t0_all)/60:.1f} 分钟")


In [ ]:
for a in ARCHS:
    f = f1_score(Y, oof[a].argmax(1), average="macro")
    print(f"OOF {a}: {f:.4f}")

blend = np.mean([oof[a] for a in ARCHS], axis=0)
print("OOF 简单平均:", f1_score(Y, blend.argmax(1), average="macro"))

A, B = ARCHS
best_f, best_w = 0.0, None
for w1 in np.arange(0, 1.01, 0.05):
    w2 = 1 - w1
    m = w1*oof[A] + w2*oof[B]
    f = f1_score(Y, m.argmax(1), average="macro")
    if f > best_f:
        best_f, best_w = f, (round(w1,2), round(w2,2))
print(f"OOF 最优集成: {best_f:.4f} 权重 {A}={best_w[0]} {B}={best_w[1]}")
WEIGHTS = dict(zip(ARCHS, best_w))


In [ ]:
class TestDS(Dataset):
    def __init__(self, files, tf): self.files=files; self.tf=tf
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        img = segment_pil(Image.open(self.files[i]))
        return self.tf(img)

TTA = [(384, False),(384, True),(352, False),(352, True)]
test_probs = {a: torch.zeros(len(test_files), len(CLASSES)) for a in ARCHS}

for fold in range(N_FOLDS):
    for arch in ARCHS:
        ckpt = f"/kaggle/working/ckpt_{arch}_f{fold}.pth"
        d = torch.load(ckpt, map_location=DEVICE, weights_only=False)
        m = build_model(arch, len(CLASSES)).to(DEVICE)
        m.load_state_dict(d["state_dict"]); m.eval()
        for size, flip in TTA:
            tfs = [transforms.Resize((size,size))]
            if flip: tfs.append(transforms.RandomHorizontalFlip(p=1.0))
            tfs += [transforms.ToTensor(), transforms.Normalize(MEAN,STD)]
            dl = DataLoader(TestDS(test_files, transforms.Compose(tfs)),
                            batch_size=BATCH, shuffle=False, num_workers=NWORK,
                            pin_memory=True)
            parts = []
            with torch.no_grad():
                for x in dl:
                    with torch.amp.autocast("cuda", enabled=AMP_ON):
                        parts.append(torch.softmax(m(x.to(DEVICE)),1).cpu())
            test_probs[arch] += torch.cat(parts)/(N_FOLDS*len(TTA))
        del m; torch.cuda.empty_cache()
        print(f"  f{fold} {arch} done", flush=True)

final = sum(WEIGHTS[a]*test_probs[a] for a in ARCHS)
preds = final.argmax(1).tolist()
print("\n预测分布:", Counter(CLASSES[p] for p in preds))


In [ ]:
file_to_label = {os.path.basename(f): CLASSES[p] for f,p in zip(test_files,preds)}
out = tmpl_df.copy()
def lookup(x):
    return file_to_label.get(x, file_to_label.get(os.path.splitext(x)[0]+".png"))
out["Category"] = out["ID"].map(lookup)

VALID = set(CLASSES)
assert out["Category"].notna().all(), "未映射ID"
assert set(out["Category"].unique()) <= VALID, "非法标签"
assert len(out)==len(tmpl_df), "行数不一致"
out.to_csv("/kaggle/working/submission.csv", index=False)
print("saved submission.csv rows:", len(out))
print(out.head())
print(out["Category"].value_counts())

for f in glob.glob("/kaggle/working/ckpt_*.pth"):
    os.remove(f)
print("\n=== DONE ===")
